# Vahan Product Analytics
## Lead-Source Cohort Performance

**Objective:**  
Evaluate lead-source cohort performance, identify the strongest cohorts, build an aggregate SQL view of the funnel, and develop a machine learning model to understand factors associated with FT conversion.

In [ ]:
!pip -q install duckdb openpyxl

## 1. Business Objective

The analysis focuses on three questions:

1. Which three lead-source cohorts perform best, and what metric should be used to evaluate them?
2. What is the appropriate aggregation level for evaluating cohort performance, and what SQL query produces this view?
3. Which factors are associated with the probability of FT conversion, and how well can an ML model distinguish FT from non-FT leads?

## 1. Business Objective

The analysis focuses on three questions:

1. Which three lead-source cohorts perform best, and what metric should be used to evaluate them?
2. What is the appropriate aggregation level for evaluating cohort performance, and what SQL query produces this view?
3. Which factors are associated with the probability of FT conversion, and how well can an ML model distinguish FT from non-FT leads?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

In [ ]:
FILE_PATH = "/content/Vahan_Case_Study (1).xlsx"

raw = pd.read_excel(
    FILE_PATH,
    sheet_name="Raw Data"
)

raw.head()


,upload_date,lead_source,candidate_phone,Uploaded Leads,Attempted,Connected,Attempt per Lead,tag_filled,Interested,OB_after_upload,OB_after_first_attempt,FT_after_upload,FT_after_first_attempt,upload_to_first_attempt_P50 (hrs),Attempted %,Attempt → Connected %,Connect → Interested %,Interested → FT_after_first_attempt %,Attempted → FT_after_upload %
0,2026-07-18,2W3W - 3WAHD - Khanna - 3W - 17 Jul,91148e89d497914604c9c248ccd2aca212fac665bdc396cd2443ccc206d216e9,1,1,1,1.0,1,0,0,0,0,0,112.0,100,100.0,0.0,NaN,0.0
1,2026-07-18,2W3W - 3WAHD - Khanna - 3W - 17 Jul,4aa4e7a4f4a3b9fa608c4c1729be6a8accc31f58b3d8b12bc02ad6c505f3f801,1,1,0,1.0,0,0,0,0,0,0,112.0,100,0.0,NaN,NaN,0.0
2,2026-07-18,2W3W - 3WAHD - Khanna - 3W - 17 Jul,0e97e34090259c1ce6cef1fd1c491fcfc469ba9d2d1ce784d5205fc523955493,1,1,0,1.0,0,0,0,0,0,0,89.0,100,0.0,NaN,NaN,0.0
3,2026-07-18,2W3W - 3WAHD - Khanna - 3W - 17 Jul,cdc003718802cffa769a478bcc0fe31aa20ac3ffe97a7c0f3c6738a2105fa08b,1,1,0,1.0,0,0,0,0,0,0,106.0,100,0.0,NaN,NaN,0.0
4,2026-07-18,2W3W - 3WAHD - Khanna - 3W - 17 Jul,d70833a770ab63e1992254dad148b100bbbadbe0e946736e9feb33252b728819,1,1,0,1.0,0,0,0,0,0,0,90.0,100,0.0,NaN,NaN,0.0


In [ ]:
print(f"Rows: {raw.shape[0]:,}")
print(f"Columns: {raw.shape[1]}")

Rows: 18,198
Columns: 19


### Initial Observation

The raw dataset contains 18,198 lead-level observations across 19 fields. The analysis will therefore be performed at the lead level initially, with aggregation to the lead-source cohort level where required by the business questions.

The raw data is retained separately so that all transformations remain traceable and reproducible.

## 2. Data Understanding

The raw dataset contains lead-level records. Each row represents a lead/candidate observation and contains information about its source, outreach funnel progression, onboarding activity and FT outcome.

The primary business dimension is `lead_source`, while `FT_after_upload` is used as the primary final conversion outcome for cohort comparison.

In [ ]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18198 entries, 0 to 18197
Data columns (total 19 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   upload_date                            18198 non-null  object 
 1   lead_source                            18198 non-null  object 
 2   candidate_phone                        18197 non-null  object 
 3   Uploaded Leads                         18198 non-null  int64  
 4   Attempted                              18198 non-null  int64  
 5   Connected                              18198 non-null  int64  
 6   Attempt per Lead                       11973 non-null  float64
 7   tag_filled                             18198 non-null  int64  
 8   Interested                             18198 non-null  int64  
 9   OB_after_upload                        18198 non-null  int64  
 10  OB_after_first_attempt                 18198 non-null  int64  
 11  FT

In [ ]:
raw.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
upload_date,18198,11,2026-07-21,5188,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lead_source,18198,16,OLX - Ashwin - 2W - 17 Jul,5182,NaN,NaN,NaN,NaN,NaN,NaN,NaN
candidate_phone,18197,17097,bf930f62383d3b24ba18191f10efeb3ed65b8eee36e4fa625b31917412ea593c,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Uploaded Leads,18198.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
Attempted,18198.0,NaN,NaN,NaN,0.657929,0.474416,0.0,0.0,1.0,1.0,1.0
Connected,18198.0,NaN,NaN,NaN,0.304979,0.460411,0.0,0.0,0.0,1.0,1.0
Attempt per Lead,11973.0,NaN,NaN,NaN,1.314541,0.594372,1.0,1.0,1.0,2.0,6.0
tag_filled,18198.0,NaN,NaN,NaN,0.301077,0.458739,0.0,0.0,0.0,1.0,1.0
Interested,18198.0,NaN,NaN,NaN,0.019123,0.136961,0.0,0.0,0.0,0.0,1.0
OB_after_upload,18198.0,NaN,NaN,NaN,0.006539,0.080603,0.0,0.0,0.0,0.0,1.0


In [ ]:
raw["lead_source"].nunique()

16

In [ ]:
raw["lead_source"].value_counts()

,count
lead_source,
OLX - Ashwin - 2W - 17 Jul,5182
AI Connected band Not Interested,2137
PreOb-Ob Fees Paid 29th Jul (set 2),1558
Khanna- 2W 26th Jul,1546
Single Referral > 7 days- 24th Jul,1500
PreOb-Ob Fees Paid 29th Jul (set 1),1483
AI Connected but not Connected by TC- Set 1,1480
AI Connected but not Connected by TC- Set 2,1193
Khanna AI,886


In [ ]:
funnel_cols = [
    "Attempted",
    "Connected",
    "tag_filled",
    "Interested",
    "OB_after_upload",
    "OB_after_first_attempt",
    "FT_after_upload",
    "FT_after_first_attempt"
]

funnel_summary = pd.DataFrame({
    "Count": raw[funnel_cols].sum(),
    "Rate_of_all_leads_%": raw[funnel_cols].mean() * 100
})

funnel_summary["Rate_of_all_leads_%"] = (
    funnel_summary["Rate_of_all_leads_%"].round(2)
)

funnel_summary

,Count,Rate_of_all_leads_%
Attempted,11973,65.79
Connected,5550,30.50
tag_filled,5479,30.11
Interested,348,1.91
OB_after_upload,119,0.65
OB_after_first_attempt,29,0.16
FT_after_upload,54,0.30
FT_after_first_attempt,17,0.09


### Funnel Overview

The dataset contains 18,198 uploaded leads. Of these, 65.79% were attempted and 30.50% were connected. However, only 1.91% progressed to the Interested stage, and 0.30% ultimately reached FT after upload.

This indicates that the major reduction in the funnel occurs after initial outreach and connection. Therefore, cohort performance should be evaluated using downstream conversion outcomes rather than lead volume alone.

In [ ]:
cohort_size = (
    raw.groupby("lead_source")
       .size()
       .sort_values(ascending=False)
       .rename("lead_count")
       .reset_index()
)

cohort_size

,lead_source,lead_count
0,OLX - Ashwin - 2W - 17 Jul,5182
1,AI Connected band Not Interested,2137
2,PreOb-Ob Fees Paid 29th Jul (set 2),1558
3,Khanna- 2W 26th Jul,1546
4,Single Referral > 7 days- 24th Jul,1500
5,PreOb-Ob Fees Paid 29th Jul (set 1),1483
6,AI Connected but not Connected by TC- Set 1,1480
7,AI Connected but not Connected by TC- Set 2,1193
8,Khanna AI,886
9,2W3W - 3WCNG - Khanna - 3W - 17 Jul,530


In [ ]:
cohort_performance = (
    raw.groupby("lead_source")
       .agg(
           uploaded_leads=("Uploaded Leads", "sum"),
           attempted=("Attempted", "sum"),
           connected=("Connected", "sum"),
           interested=("Interested", "sum"),
           ob_after_upload=("OB_after_upload", "sum"),
           ft_after_upload=("FT_after_upload", "sum")
       )
       .reset_index()
)

cohort_performance["attempt_rate_%"] = (
    cohort_performance["attempted"]
    / cohort_performance["uploaded_leads"]
    * 100
)

cohort_performance["connect_rate_%"] = (
    cohort_performance["connected"]
    / cohort_performance["attempted"].replace(0, np.nan)
    * 100
)

cohort_performance["interest_rate_%"] = (
    cohort_performance["interested"]
    / cohort_performance["connected"].replace(0, np.nan)
    * 100
)

cohort_performance["ft_conversion_%"] = (
    cohort_performance["ft_after_upload"]
    / cohort_performance["uploaded_leads"]
    * 100
)

cohort_performance = cohort_performance.sort_values(
    "ft_conversion_%",
    ascending=False
)

cohort_performance

,lead_source,uploaded_leads,attempted,connected,interested,ob_after_upload,ft_after_upload,attempt_rate_%,connect_rate_%,interest_rate_%,ft_conversion_%
15,Single Referral > 7 days- 24th Jul,1500,1454,690,7,30,14,96.933333,47.455296,1.014493,0.933333
11,Khanna- 2W 26th Jul,1546,1376,574,20,39,14,89.003881,41.715116,3.484321,0.905563
13,PreOb-Ob Fees Paid 29th Jul (set 1),1483,1433,650,103,15,7,96.628456,45.359386,15.846154,0.472016
14,PreOb-Ob Fees Paid 29th Jul (set 2),1558,1519,735,122,14,7,97.496791,48.387097,16.598639,0.449294
6,AI Connected but not Connected by TC- Set 1,1480,1331,694,29,9,5,89.932432,52.141247,4.178674,0.337838
7,AI Connected but not Connected by TC- Set 2,1193,1066,466,26,6,2,89.354568,43.714822,5.579399,0.167645
10,Khanna AI,886,245,137,2,3,1,27.652370,55.918367,1.459854,0.112867
12,OLX - Ashwin - 2W - 17 Jul,5182,962,372,9,3,4,18.564261,38.669439,2.419355,0.077190
4,50K 2W5 - Khanna - 2W - 17 Jul,1,0,0,0,0,0,0.000000,NaN,NaN,0.000000
5,AI Connected band Not Interested,2137,1359,637,21,0,0,63.593823,46.872701,3.296703,0.000000


In [ ]:
ft_comparison = (
    raw.groupby("lead_source")
       .agg(
           uploaded_leads=("Uploaded Leads", "sum"),
           ft_after_upload=("FT_after_upload", "sum"),
           ft_after_first_attempt=("FT_after_first_attempt", "sum")
       )
       .reset_index()
)

ft_comparison["ft_after_upload_rate_%"] = (
    ft_comparison["ft_after_upload"]
    / ft_comparison["uploaded_leads"]
    * 100
)

ft_comparison["ft_after_first_attempt_rate_%"] = (
    ft_comparison["ft_after_first_attempt"]
    / ft_comparison["uploaded_leads"]
    * 100
)

ft_comparison = ft_comparison.sort_values(
    "ft_after_upload_rate_%",
    ascending=False
)

ft_comparison

,lead_source,uploaded_leads,ft_after_upload,ft_after_first_attempt,ft_after_upload_rate_%,ft_after_first_attempt_rate_%
15,Single Referral > 7 days- 24th Jul,1500,14,2,0.933333,0.133333
11,Khanna- 2W 26th Jul,1546,14,6,0.905563,0.388098
13,PreOb-Ob Fees Paid 29th Jul (set 1),1483,7,2,0.472016,0.134862
14,PreOb-Ob Fees Paid 29th Jul (set 2),1558,7,5,0.449294,0.320924
6,AI Connected but not Connected by TC- Set 1,1480,5,0,0.337838,0.000000
7,AI Connected but not Connected by TC- Set 2,1193,2,0,0.167645,0.000000
10,Khanna AI,886,1,0,0.112867,0.000000
12,OLX - Ashwin - 2W - 17 Jul,5182,4,2,0.077190,0.038595
4,50K 2W5 - Khanna - 2W - 17 Jul,1,0,0,0.000000,0.000000
5,AI Connected band Not Interested,2137,0,0,0.000000,0.000000


In [ ]:
ft_relationship = pd.crosstab(
    raw["FT_after_upload"],
    raw["FT_after_first_attempt"]
)

ft_relationship

FT_after_first_attempt,0,1
FT_after_upload,,
0,18144,0
1,37,17


In [ ]:
date_summary = (
    raw.groupby("upload_date")
       .agg(
           uploaded_leads=("Uploaded Leads", "sum"),
           ft_after_upload=("FT_after_upload", "sum")
       )
       .reset_index()
       .sort_values("upload_date")
)

date_summary["ft_conversion_%"] = (
    date_summary["ft_after_upload"]
    / date_summary["uploaded_leads"]
    * 100
)

date_summary

,upload_date,uploaded_leads,ft_after_upload,ft_conversion_%
0,2026-07-18,1227,0,0.000000
1,2026-07-21,5188,4,0.077101
2,2026-07-25,1500,14,0.933333
3,2026-07-27,1546,14,0.905563
4,2026-07-29,1483,7,0.472016
5,2026-07-30,1558,7,0.449294
6,2026-08-01,1480,5,0.337838
7,2026-08-03,1193,2,0.167645
8,2026-08-04,1537,0,0.000000
9,2026-08-05,600,0,0.000000


In [ ]:
source_date_check = (
    raw.groupby("lead_source")["upload_date"]
       .agg(["nunique", "min", "max"])
       .sort_values("nunique", ascending=False)
)

source_date_check

,nunique,min,max
lead_source,,,
AI Connected band Not Interested,2,2026-08-04,2026-08-05
2W3W - 3WAHD - Khanna - 3W - 17 Jul,1,2026-07-18,2026-07-18
2W3W - 3WEV - Khanna - 3W - 17 Jul,1,2026-07-18,2026-07-18
2W3W - 3WCNG - Khanna - 3W - 17 Jul,1,2026-07-18,2026-07-18
50K 2W4 - Khanna - 2W - 17 Jul,1,2026-07-18,2026-07-18
50K 2W5 - Khanna - 2W - 17 Jul,1,2026-07-21,2026-07-21
AI Connected but not Connected by TC- Set 1,1,2026-08-01,2026-08-01
AI Connected but not Connected by TC- Set 2,1,2026-08-03,2026-08-03
JobHai - Khanna - Riders - 16 Jul,1,2026-07-21,2026-07-21


In [ ]:
latest_date = pd.to_datetime(raw["upload_date"]).max()

cohort_maturity = (
    raw.groupby("lead_source")
       .agg(
           first_upload_date=("upload_date", "min"),
           last_upload_date=("upload_date", "max"),
           uploaded_leads=("Uploaded Leads", "sum"),
           ft_after_upload=("FT_after_upload", "sum")
       )
       .reset_index()
)

cohort_maturity["first_upload_date"] = pd.to_datetime(
    cohort_maturity["first_upload_date"]
)

cohort_maturity["last_upload_date"] = pd.to_datetime(
    cohort_maturity["last_upload_date"]
)

cohort_maturity["days_observed"] = (
    latest_date - cohort_maturity["last_upload_date"]
).dt.days

cohort_maturity["ft_conversion_%"] = (
    cohort_maturity["ft_after_upload"]
    / cohort_maturity["uploaded_leads"]
    * 100
)

cohort_maturity.sort_values("first_upload_date")

,lead_source,first_upload_date,last_upload_date,uploaded_leads,ft_after_upload,days_observed,ft_conversion_%
0,2W3W - 3WAHD - Khanna - 3W - 17 Jul,2026-07-18,2026-07-18,495,0,19,0.000000
1,2W3W - 3WCNG - Khanna - 3W - 17 Jul,2026-07-18,2026-07-18,530,0,19,0.000000
2,2W3W - 3WEV - Khanna - 3W - 17 Jul,2026-07-18,2026-07-18,201,0,19,0.000000
3,50K 2W4 - Khanna - 2W - 17 Jul,2026-07-18,2026-07-18,1,0,19,0.000000
4,50K 2W5 - Khanna - 2W - 17 Jul,2026-07-21,2026-07-21,1,0,16,0.000000
8,JobHai - Khanna - Riders - 16 Jul,2026-07-21,2026-07-21,2,0,16,0.000000
9,JobHai - Khanna - Riders - 17 Jul,2026-07-21,2026-07-21,3,0,16,0.000000
12,OLX - Ashwin - 2W - 17 Jul,2026-07-21,2026-07-21,5182,4,16,0.077190
15,Single Referral > 7 days- 24th Jul,2026-07-25,2026-07-25,1500,14,12,0.933333
11,Khanna- 2W 26th Jul,2026-07-27,2026-07-27,1546,14,10,0.905563


In [ ]:
raw["upload_to_first_attempt_P50 (hrs)"].describe()

,upload_to_first_attempt_P50 (hrs)
count,12932.000000
mean,21.000773
std,81.429831
min,-611.000000
25%,12.000000
50%,16.000000
75%,34.000000
max,457.500000


In [ ]:
raw.loc[
    raw["upload_to_first_attempt_P50 (hrs)"] < 0,
    ["upload_date", "lead_source", "upload_to_first_attempt_P50 (hrs)"]
].head(20)

,upload_date,lead_source,upload_to_first_attempt_P50 (hrs)
1232,2026-07-21,JobHai - Khanna - Riders - 17 Jul,-8.0
1234,2026-07-21,OLX - Ashwin - 2W - 17 Jul,-9.0
1265,2026-07-21,OLX - Ashwin - 2W - 17 Jul,-9.0
1279,2026-07-21,OLX - Ashwin - 2W - 17 Jul,-10.0
1318,2026-07-21,OLX - Ashwin - 2W - 17 Jul,-8.0
1376,2026-07-21,OLX - Ashwin - 2W - 17 Jul,-8.0
1387,2026-07-21,OLX - Ashwin - 2W - 17 Jul,-110.0
1406,2026-07-21,OLX - Ashwin - 2W - 17 Jul,-6.0
1414,2026-07-21,OLX - Ashwin - 2W - 17 Jul,-7.0
1453,2026-07-21,OLX - Ashwin - 2W - 17 Jul,-10.0


In [ ]:
timing_check = pd.Series({
    "total_non_null": raw["upload_to_first_attempt_P50 (hrs)"].notna().sum(),
    "negative_values": (raw["upload_to_first_attempt_P50 (hrs)"] < 0).sum(),
    "zero_values": (raw["upload_to_first_attempt_P50 (hrs)"] == 0).sum(),
    "positive_values": (raw["upload_to_first_attempt_P50 (hrs)"] > 0).sum()
})

timing_check

,0
total_non_null,12932
negative_values,1496
zero_values,3
positive_values,11433


In [ ]:
overall_ft = pd.DataFrame({
    "uploaded_leads": [raw["Uploaded Leads"].sum()],
    "ft_after_upload": [raw["FT_after_upload"].sum()]
})

overall_ft["ft_conversion_%"] = (
    overall_ft["ft_after_upload"]
    / overall_ft["uploaded_leads"]
    * 100
)

overall_ft

,uploaded_leads,ft_after_upload,ft_conversion_%
0,18198,54,0.296736


In [ ]:
cohort_ranking = cohort_performance[
    [
        "lead_source",
        "uploaded_leads",
        "ft_after_upload",
        "ft_conversion_%"
    ]
].copy()

cohort_ranking["vs_overall_ft_rate_pp"] = (
    cohort_ranking["ft_conversion_%"] - overall_ft.loc[0, "ft_conversion_%"]
)

cohort_ranking.sort_values(
    "ft_conversion_%",
    ascending=False
)

,lead_source,uploaded_leads,ft_after_upload,ft_conversion_%,vs_overall_ft_rate_pp
15,Single Referral > 7 days- 24th Jul,1500,14,0.933333,0.636597
11,Khanna- 2W 26th Jul,1546,14,0.905563,0.608827
13,PreOb-Ob Fees Paid 29th Jul (set 1),1483,7,0.472016,0.175280
14,PreOb-Ob Fees Paid 29th Jul (set 2),1558,7,0.449294,0.152558
6,AI Connected but not Connected by TC- Set 1,1480,5,0.337838,0.041102
7,AI Connected but not Connected by TC- Set 2,1193,2,0.167645,-0.129091
10,Khanna AI,886,1,0.112867,-0.183869
12,OLX - Ashwin - 2W - 17 Jul,5182,4,0.077190,-0.219546
4,50K 2W5 - Khanna - 2W - 17 Jul,1,0,0.000000,-0.296736
5,AI Connected band Not Interested,2137,0,0.000000,-0.296736


In [ ]:
cohort_ranking["ft_share_%"] = (
    cohort_ranking["ft_after_upload"]
    / overall_ft.loc[0, "ft_after_upload"]
    * 100
)

cohort_ranking.sort_values(
    "ft_after_upload",
    ascending=False
)

,lead_source,uploaded_leads,ft_after_upload,ft_conversion_%,vs_overall_ft_rate_pp,ft_share_%
15,Single Referral > 7 days- 24th Jul,1500,14,0.933333,0.636597,25.925926
11,Khanna- 2W 26th Jul,1546,14,0.905563,0.608827,25.925926
13,PreOb-Ob Fees Paid 29th Jul (set 1),1483,7,0.472016,0.175280,12.962963
14,PreOb-Ob Fees Paid 29th Jul (set 2),1558,7,0.449294,0.152558,12.962963
6,AI Connected but not Connected by TC- Set 1,1480,5,0.337838,0.041102,9.259259
12,OLX - Ashwin - 2W - 17 Jul,5182,4,0.077190,-0.219546,7.407407
7,AI Connected but not Connected by TC- Set 2,1193,2,0.167645,-0.129091,3.703704
10,Khanna AI,886,1,0.112867,-0.183869,1.851852
4,50K 2W5 - Khanna - 2W - 17 Jul,1,0,0.000000,-0.296736,0.000000
5,AI Connected band Not Interested,2137,0,0.000000,-0.296736,0.000000


In [ ]:
top_3_ft = (
    cohort_ranking
    .sort_values("ft_conversion_%", ascending=False)
    .head(3)["ft_after_upload"]
    .sum()
)

top_3_ft_share = (
    top_3_ft / overall_ft.loc[0, "ft_after_upload"] * 100
)

top_3_ft, top_3_ft_share

(np.int64(35), np.float64(64.81481481481481))

In [66]:
cohort_q1 = (
    raw.groupby("lead_source")
       .agg(
           uploaded_leads=("Uploaded Leads", "sum"),
           ft_after_upload=("FT_after_upload", "sum")
       )
       .reset_index()
)

cohort_q1["ft_conversion_%"] = (
    cohort_q1["ft_after_upload"]
    / cohort_q1["uploaded_leads"]
    * 100
)

cohort_q1 = cohort_q1[
    cohort_q1["uploaded_leads"] >= 100
].sort_values(
    "ft_conversion_%",
    ascending=False
)

cohort_q1.head(3)

,lead_source,uploaded_leads,ft_after_upload,ft_conversion_%
15,Single Referral > 7 days- 24th Jul,1500,14,0.933333
11,Khanna- 2W 26th Jul,1546,14,0.905563
13,PreOb-Ob Fees Paid 29th Jul (set 1),1483,7,0.472016


In [67]:
overall_ft_rate = (
    raw["FT_after_upload"].sum()
    / raw["Uploaded Leads"].sum()
    * 100
)

q1_top3 = cohort_q1.head(3).copy()

q1_top3["vs_overall_pp"] = (
    q1_top3["ft_conversion_%"]
    - overall_ft_rate
)

q1_top3["vs_overall_x"] = (
    q1_top3["ft_conversion_%"]
    / overall_ft_rate
)

q1_top3[
    [
        "lead_source",
        "uploaded_leads",
        "ft_after_upload",
        "ft_conversion_%",
        "vs_overall_pp",
        "vs_overall_x"
    ]
]

,lead_source,uploaded_leads,ft_after_upload,ft_conversion_%,vs_overall_pp,vs_overall_x
15,Single Referral > 7 days- 24th Jul,1500,14,0.933333,0.636597,3.145333
11,Khanna- 2W 26th Jul,1546,14,0.905563,0.608827,3.051746
13,PreOb-Ob Fees Paid 29th Jul (set 1),1483,7,0.472016,0.175280,1.590695


### Q1 — Best 3 Cohorts

FT after upload conversion rate was selected as the primary metric because FT is the final business outcome and the rate normalizes for differences in cohort size.

To avoid unstable rankings from extremely small cohorts, cohorts with fewer than 100 uploaded leads were excluded from the ranking.

The top 3 cohorts are:

1. **Single Referral > 7 days- 24th Jul**
   - FT conversion: **0.933%**
   - 14 FT conversions from 1,500 leads
   - **3.15×** the overall FT conversion rate

2. **Khanna- 2W 26th Jul**
   - FT conversion: **0.906%**
   - 14 FT conversions from 1,546 leads
   - **3.05×** the overall FT conversion rate

3. **PreOb-Ob Fees Paid 29th Jul (set 1)**
   - FT conversion: **0.472%**
   - 7 FT conversions from 1,483 leads
   - **1.59×** the overall FT conversion rate

The overall FT after upload conversion rate is **0.297%**.

The strongest-performing cohort was Single Referral > 7 days- 24th Jul, followed closely by Khanna- 2W 26th Jul. Both cohorts achieved more than 3× the overall FT conversion rate, while PreOb-Ob Fees Paid 29th Jul (set 1) also performed above the overall benchmark.

## Question 2 — Cohort-level Funnel Aggregation

The lead-level data is aggregated by `lead_source` to evaluate cohort performance across the acquisition and conversion funnel.

The aggregation reports lead volume, funnel progression, and FT conversion. Rates are calculated from aggregated counts rather than averaging row-level percentages, which avoids distortion from missing or zero-denominator records.

In [68]:
q2_sql = """
SELECT
    lead_source,

    COUNT(*) AS uploaded_leads,

    SUM(Attempted) AS attempted,
    SUM(Connected) AS connected,
    SUM(tag_filled) AS tag_filled,
    SUM(Interested) AS interested,

    SUM(OB_after_upload) AS ob_after_upload,
    SUM(OB_after_first_attempt) AS ob_after_first_attempt,

    SUM(FT_after_upload) AS ft_after_upload,
    SUM(FT_after_first_attempt) AS ft_after_first_attempt,

    ROUND(
        100.0 * SUM(Attempted) / NULLIF(COUNT(*), 0),
        2
    ) AS attempted_pct,

    ROUND(
        100.0 * SUM(Connected) / NULLIF(SUM(Attempted), 0),
        2
    ) AS attempt_to_connected_pct,

    ROUND(
        100.0 * SUM(Interested) / NULLIF(SUM(Connected), 0),
        2
    ) AS connect_to_interested_pct,

    ROUND(
        100.0 * SUM(FT_after_upload) / NULLIF(COUNT(*), 0),
        2
    ) AS ft_after_upload_conversion_pct

FROM raw

GROUP BY lead_source

ORDER BY ft_after_upload_conversion_pct DESC;
"""

q2_result = con.execute(q2_sql).df()

q2_result

,lead_source,uploaded_leads,attempted,connected,tag_filled,interested,ob_after_upload,ob_after_first_attempt,ft_after_upload,ft_after_first_attempt,attempted_pct,attempt_to_connected_pct,connect_to_interested_pct,ft_after_upload_conversion_pct
0,Single Referral > 7 days- 24th Jul,1500,1454.0,690.0,676.0,7.0,30.0,4.0,14.0,2.0,96.93,47.46,1.01,0.93
1,Khanna- 2W 26th Jul,1546,1376.0,574.0,561.0,20.0,39.0,10.0,14.0,6.0,89.00,41.72,3.48,0.91
2,PreOb-Ob Fees Paid 29th Jul (set 1),1483,1433.0,650.0,645.0,103.0,15.0,6.0,7.0,2.0,96.63,45.36,15.85,0.47
3,PreOb-Ob Fees Paid 29th Jul (set 2),1558,1519.0,735.0,728.0,122.0,14.0,6.0,7.0,5.0,97.50,48.39,16.60,0.45
4,AI Connected but not Connected by TC- Set 1,1480,1331.0,694.0,684.0,29.0,9.0,2.0,5.0,0.0,89.93,52.14,4.18,0.34
5,AI Connected but not Connected by TC- Set 2,1193,1066.0,466.0,464.0,26.0,6.0,0.0,2.0,0.0,89.35,43.71,5.58,0.17
6,Khanna AI,886,245.0,137.0,136.0,2.0,3.0,0.0,1.0,0.0,27.65,55.92,1.46,0.11
7,OLX - Ashwin - 2W - 17 Jul,5182,962.0,372.0,364.0,9.0,3.0,1.0,4.0,2.0,18.56,38.67,2.42,0.08
8,2W3W - 3WEV - Khanna - 3W - 17 Jul,201,201.0,91.0,90.0,1.0,0.0,0.0,0.0,0.0,100.00,45.27,1.10,0.00
9,50K 2W5 - Khanna - 2W - 17 Jul,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,NaN,NaN,0.00


### Q2 — Cohort-level Funnel Insights

The SQL aggregation shows substantial variation in funnel performance across lead sources.

- **Single Referral > 7 days- 24th Jul** has the highest FT conversion at **0.93%**, with 14 FT conversions from 1,500 uploaded leads.
- **Khanna- 2W 26th Jul** follows at **0.91%**, also producing 14 FT conversions.
- The two **PreOb-Ob Fees Paid** cohorts show strong mid-funnel engagement, with Connect → Interested rates of **15.85%** and **16.60%**, but their final FT conversion is lower at **0.47%** and **0.45%**.
- **OLX - Ashwin - 2W - 17 Jul** contributes the largest lead volume with 5,182 leads, but its FT conversion is only **0.08%**. This indicates that high lead volume does not necessarily translate into high FT efficiency.
- **Khanna AI** has a relatively high Attempt → Connected rate of **55.92%**, but only **0.11%** FT conversion, indicating that connection quality alone does not guarantee downstream FT.
- Several small cohorts have very few leads and zero FT conversions. These should not be interpreted as evidence of poor cohort quality without sufficient sample size.

Overall, the SQL analysis shows that **cohort quality varies significantly across the funnel**, and lead volume should be evaluated together with downstream FT conversion rather than as a standalone success metric.

## Question 3 — Predicting FT After Upload

The objective is to build an interpretable machine learning model that identifies factors associated with the likelihood of FT after upload.

The target variable is `FT_after_upload`.

Because FT occurs for only 54 of 18,198 leads (0.297%), the problem is highly imbalanced. Therefore, model performance is evaluated using ROC-AUC and PR-AUC rather than accuracy alone.

For the final model, the funnel is represented using a single `funnel_stage` variable, indicating the deepest observed stage reached by a lead:

- 0 — Not attempted
- 1 — Attempted
- 2 — Connected
- 3 — Tag filled
- 4 — Interested

`lead_source` is retained because cohort/source characteristics show substantial differences in historical FT conversion.

Downstream FT and OB outcome variables are excluded from the predictors to avoid target leakage.

In [69]:
final_features = [
    "lead_source",
    "funnel_stage"
]

X_final = raw[final_features].copy()
y_final = raw["FT_after_upload"].copy()

print("Features:", final_features)
print("X shape:", X_final.shape)
print("y shape:", y_final.shape)

print("\nTarget distribution:")
print(y_final.value_counts())

Features: ['lead_source', 'funnel_stage']
X shape: (18198, 2)
y shape: (18198,)

Target distribution:
FT_after_upload
0    18144
1       54
Name: count, dtype: int64


### Train-Test Split

An 80/20 stratified split is used so that the rare FT class is represented in both training and test sets.

Stratification is important here because only 54 of the 18,198 observations are positive FT outcomes.

In [71]:
X_final_train, X_final_test, y_final_train, y_final_test = train_test_split(
    X_final,
    y_final,
    test_size=0.20,
    stratify=y_final,
    random_state=42
)

print("Training set:", X_final_train.shape)
print("Test set:", X_final_test.shape)

print("\nTraining target distribution:")
print(y_final_train.value_counts())

print("\nTest target distribution:")
print(y_final_test.value_counts())

Training set: (14558, 2)
Test set: (3640, 2)

Training target distribution:
FT_after_upload
0    14515
1       43
Name: count, dtype: int64

Test target distribution:
FT_after_upload
0    3629
1      11
Name: count, dtype: int64


### Logistic Regression

Logistic Regression is selected as the model because it provides an interpretable relationship between the input features and FT outcome.

Class weighting is used to address the severe imbalance between FT and non-FT leads. The model is therefore optimized to identify the rare FT class rather than simply predicting the majority class.

In [72]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

final_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            ["lead_source"]
        ),
        (
            "numeric",
            "passthrough",
            ["funnel_stage"]
        )
    ]
)

final_model = Pipeline(
    steps=[
        ("preprocessor", final_preprocessor),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

final_model.fit(
    X_final_train,
    y_final_train
)

print("Final model trained successfully.")

Final model trained successfully.


### Model Performance

Because FT is a highly imbalanced outcome, both ROC-AUC and PR-AUC are reported.

ROC-AUC measures the model's ability to rank FT leads above non-FT leads across thresholds, while PR-AUC is particularly informative for rare positive outcomes.

In [73]:
from sklearn.metrics import roc_auc_score, average_precision_score

final_proba = final_model.predict_proba(
    X_final_test
)[:, 1]

final_roc_auc = roc_auc_score(
    y_final_test,
    final_proba
)

final_pr_auc = average_precision_score(
    y_final_test,
    final_proba
)

print(f"ROC-AUC: {final_roc_auc:.4f}")
print(f"PR-AUC:  {final_pr_auc:.4f}")

ROC-AUC: 0.7815
PR-AUC:  0.0130


/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


### Model Performance — Interpretation

The final Logistic Regression model achieved a ROC-AUC of **0.7815** and a PR-AUC of **0.0130** on the stratified holdout set.

Given the very low FT base rate of **0.297%**, PR-AUC is particularly important because it focuses on performance for the rare positive class.

The model therefore demonstrates useful ranking ability, but the low absolute PR-AUC and severe class imbalance mean that its output should be used primarily for **lead prioritization rather than as a calibrated probability of FT**.

### Confusion Matrix

A threshold of 0.5 is used as a transparent reference point for binary classification.

Because FT is extremely rare, accuracy alone is not considered an appropriate measure of model effectiveness.

In [74]:
from sklearn.metrics import confusion_matrix, classification_report

final_threshold = 0.5

final_pred = (
    final_proba >= final_threshold
).astype(int)

final_cm = confusion_matrix(
    y_final_test,
    final_pred
)

print("Confusion Matrix:")
print(final_cm)

print("\nClassification Report:")
print(
    classification_report(
        y_final_test,
        final_pred,
        digits=4,
        zero_division=0
    )
)

Confusion Matrix:
[[2483 1146]
 [   4    7]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9984    0.6842    0.8120      3629
           1     0.0061    0.6364    0.0120        11

    accuracy                         0.6841      3640
   macro avg     0.5022    0.6603    0.4120      3640
weighted avg     0.9954    0.6841    0.8096      3640



### Confusion Matrix — Interpretation

At the 0.5 classification threshold:

- **True Positives:** 7
- **False Negatives:** 4
- **True Negatives:** 2,483
- **False Positives:** 1,146

The model identifies **7 of the 11 FT leads**, giving an FT recall of **63.64%**. However, precision for the FT class is only **0.61%**, because the positive class is extremely rare and the model produces a large number of false positives.

The overall accuracy of **68.41% is not used as the primary performance measure**, since a highly imbalanced dataset can make accuracy misleading.

Therefore, the model is better suited for **ranking and prioritizing leads** than for making a strict FT/non-FT classification at a fixed threshold.

In [75]:
final_model_coefficients = final_model.named_steps["model"].coef_[0]

feature_names = final_model.named_steps[
    "preprocessor"
].get_feature_names_out()

coefficient_summary = pd.DataFrame({
    "feature": feature_names,
    "coefficient": final_model_coefficients
})

coefficient_summary["odds_ratio"] = np.exp(
    coefficient_summary["coefficient"]
)

coefficient_summary["abs_coefficient"] = (
    coefficient_summary["coefficient"].abs()
)

coefficient_summary = coefficient_summary.sort_values(
    "abs_coefficient",
    ascending=False
)

coefficient_summary

,feature,coefficient,odds_ratio,abs_coefficient
8,categorical__lead_source_Khanna- 2W 26th Jul,3.763962,43.118938,3.763962
12,categorical__lead_source_Single Referral > 7 days- 24th Jul,3.746901,42.389515,3.746901
10,categorical__lead_source_PreOb-Ob Fees Paid 29th Jul (set 1),2.947977,19.067334,2.947977
3,categorical__lead_source_AI Connected band Not Interested,-2.815974,0.059846,2.815974
4,categorical__lead_source_AI Connected but not Connected by TC- Set 1,2.696768,14.831722,2.696768
0,categorical__lead_source_2W3W - 3WCNG - Khanna - 3W - 17 Jul,-2.467747,0.084776,2.467747
11,categorical__lead_source_PreOb-Ob Fees Paid 29th Jul (set 2),2.279188,9.768745,2.279188
5,categorical__lead_source_AI Connected but not Connected by TC- Set 2,2.213154,9.144513,2.213154
1,categorical__lead_source_2W3W - 3WEV - Khanna - 3W - 17 Jul,-2.057763,0.127739,2.057763
7,categorical__lead_source_Khanna AI,2.025222,7.577792,2.025222


### Factors Influencing FT

The final Logistic Regression model identifies `lead_source` and `funnel_stage` as the main predictive factors.

- **Lead source is the strongest source of variation in the model.** Khanna- 2W 26th Jul and Single Referral > 7 days- 24th Jul have the strongest positive coefficients, consistent with their high observed FT conversion rates of **0.906%** and **0.933%** respectively.
- **PreOb-Ob Fees Paid 29th Jul (set 1)** also has a strong positive model association and an observed FT conversion of **0.472%**.
- `funnel_stage` has a **positive coefficient (0.606)**, indicating that deeper progression through the observed funnel is associated with higher FT propensity in the model.
- Sources such as **AI Connected band Not Interested** and **2W3W - 3WCNG - Khanna - 3W - 17 Jul** have negative coefficients relative to the model's reference cohort.
- The model coefficients represent associations rather than causal effects. In addition, class weighting was used because FT is rare, so the reported odds ratios should not be interpreted as direct real-world probability multipliers.

Overall, the model indicates that **cohort/source quality and funnel progression are the strongest observable signals associated with FT**.

### Cohort-Level Validation

A second validation was performed using lead source as the grouping variable. Entire cohorts were kept together during cross-validation so that the model was evaluated on lead-source cohorts not seen during training.

This provides a more realistic estimate of how well the model may generalize to a new acquisition cohort.

In [76]:
group_validation = cross_validate(
    final_model,
    X_final,
    y_final,
    groups=raw["lead_source"],
    cv=group_cv,
    scoring={
        "roc_auc": "roc_auc",
        "pr_auc": "average_precision"
    },
    n_jobs=-1
)

print(
    "Mean ROC-AUC:",
    round(group_validation["test_roc_auc"].mean(), 4)
)

print(
    "Std ROC-AUC:",
    round(group_validation["test_roc_auc"].std(), 4)
)

print(
    "Mean PR-AUC:",
    round(group_validation["test_pr_auc"].mean(), 4)
)

print(
    "Std PR-AUC:",
    round(group_validation["test_pr_auc"].std(), 4)
)

Mean ROC-AUC: 0.6471
Std ROC-AUC: 0.0729
Mean PR-AUC: 0.0064
Std PR-AUC: 0.0035


### Cohort-Level Validation — Interpretation

The model achieved:

- Mean ROC-AUC: **0.6471 ± 0.0729**
- Mean PR-AUC: **0.0064 ± 0.0035**

This is materially lower than the random holdout performance of ROC-AUC **0.7815** and PR-AUC **0.0130**.

The reduction indicates that a meaningful portion of the predictive signal is associated with the characteristics of the observed lead-source cohorts. Generalization to a completely new cohort is therefore more difficult.

This validation provides a more conservative estimate of expected performance when the model encounters a lead-source cohort that was not represented during training.

### Q3 — Final Conclusion

The final Funnel-stage Logistic Regression model uses `lead_source` and `funnel_stage` to estimate relative FT propensity.

The model achieved a ROC-AUC of **0.7815** and PR-AUC of **0.0130** on the stratified holdout set. At a 0.5 threshold, it identified **7 of 11 FT leads**, corresponding to **63.64% recall**, but precision was only **0.61%** due to the extreme class imbalance.

The model is therefore more appropriate as a **lead-ranking and prioritization tool** rather than a calibrated FT probability model or a strict binary classifier.

The strongest observed signals are lead-source quality and funnel progression. However, cohort-level validation reduced ROC-AUC to **0.6471**, showing that generalization to unseen lead-source cohorts remains a significant limitation.

The model should therefore be used to support prioritization decisions alongside ongoing monitoring of new cohort performance, rather than as a standalone decision rule.

# Executive Summary

The analysis evaluates lead-source cohorts, funnel performance, and the ability to predict FT after upload.

### Key findings

**1. Cohort performance varies substantially**

The overall FT after upload conversion rate is **0.297%**.

The three best-performing cohorts, considering cohorts with at least 100 uploaded leads, are:

- **Single Referral > 7 days- 24th Jul:** 0.933% FT conversion
- **Khanna- 2W 26th Jul:** 0.906% FT conversion
- **PreOb-Ob Fees Paid 29th Jul (set 1):** 0.472% FT conversion

The first two cohorts perform at approximately **3× the overall FT conversion rate**.

**2. Lead volume alone is not a measure of cohort quality**

OLX - Ashwin - 2W - 17 Jul contains **5,182 leads**, the largest cohort, but produces only **4 FT conversions**, corresponding to a **0.077% FT conversion rate**.

In contrast, Single Referral > 7 days- 24th Jul produces **14 FT conversions from 1,500 leads**.

This indicates that acquisition volume should be evaluated together with downstream conversion quality.

**3. Funnel progression is associated with FT**

The funnel-stage analysis shows higher observed FT rates at deeper stages:

- Stage 0 — 0.000%
- Stage 1 — 0.343%
- Stage 2 — 0.000%
- Stage 3 — 0.604%
- Stage 4 — 0.287%

The stage-level estimates are based on a very small number of FT outcomes, so they should be interpreted as directional rather than causal.

**4. FT prediction is challenging because of extreme class imbalance**

Only **54 of 18,198 leads (0.297%)** resulted in FT after upload.

The final Funnel-stage Logistic Regression achieved:

- **ROC-AUC: 0.7815**
- **PR-AUC: 0.0130**
- **FT recall: 63.64%**
- **FT precision: 0.61%**

The confusion matrix shows that the model identifies a useful portion of the rare FT cases, but also produces many false positives.

**5. Generalization to unseen cohorts is weaker**

When entire lead-source cohorts are kept together during validation, performance falls to:

- **ROC-AUC: 0.6471 ± 0.0729**
- **PR-AUC: 0.0064 ± 0.0035**

This suggests that some of the predictive signal is cohort-specific and that performance on completely new lead sources may be substantially lower than random holdout performance.

### Overall conclusion

The strongest business signal comes from **lead-source quality combined with funnel progression**. The analysis supports prioritizing high-performing cohorts and using the ML model as a **lead-ranking tool**, while avoiding reliance on the model as a standalone binary decision system.

# Business Recommendations

1. **Prioritize high-performing acquisition cohorts**
   
   Single Referral > 7 days- 24th Jul and Khanna- 2W 26th Jul substantially outperform the overall FT conversion rate and should be investigated for the characteristics that make them effective.

2. **Do not optimize purely for lead volume**
   
   High-volume sources such as OLX generate many leads but relatively few FT outcomes. Source evaluation should incorporate downstream FT conversion.

3. **Use funnel progression for prioritization**
   
   Leads reaching deeper funnel stages generally show stronger FT association. These leads can receive higher priority for follow-up.

4. **Use the ML model for ranking rather than hard classification**
   
   Because FT is extremely rare, the model generates many false positives at a standard threshold. Ranking leads by predicted FT propensity is more appropriate.

5. **Monitor performance on new cohorts**
   
   The drop in cohort-level validation performance indicates that the model should be retrained and monitored as new lead sources and campaigns are introduced.

6. **Collect more positive FT examples**
   
   With only 54 FT outcomes, statistical uncertainty remains high. More FT observations would improve model stability, evaluation reliability, and the ability to detect meaningful cohort differences.